In [ ]:
import numpy as np
import bacco
import matplotlib.pyplot as plt

In [ ]:
import os
os.chdir("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
def pars(i, mstar):

    arr = np.vstack( ( mstar, np.ones(len(mstar)) * wind_en[i],\
                        np.ones(len(mstar)) * wind_vel[i],\
                        np.ones(len(mstar)) * rho_rec[i],\
                        np.ones(len(mstar)) * sf_ts[i],\
                        np.ones(len(mstar)) * ef_kin[i],\
                        np.ones(len(mstar)) * ef_high[i],\
                        np.ones(len(mstar)) * f_re[i])).T

    return arr

In [ ]:
wind_en_or      = []
wind_vel_or     = []
rho_rec_or      = []
sf_ts_or        = []
ef_kin_or       = []
ef_high_or      = []
f_re_or         = []

for i in range(31):
    if i<30:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro_{:d}.txt".format(i)
    else:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro.txt"

    with open(filename, 'r') as f:
        for line in f.readlines():
            if len(line.split())!=0:
                if line.split()[0] == 'WindEnergyIn1e51erg':
                    wind_en_or.append(float(line.split()[1]))
                if line.split()[0] == 'VariableWindVelFactor':
                    wind_vel_or.append(float(line.split()[1]))
                if line.split()[0] == 'WindFreeTravelDensFac':
                    rho_rec_or.append(float(line.split()[1]))
                if line.split()[0] == 'MaxSfrTimescale':
                    sf_ts_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackFactor':
                    ef_kin_or.append(float(line.split()[1]))
                if line.split()[0] == 'BlackHoleFeedbackFactor':
                    ef_high_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackReiorientationFactor':
                    f_re_or.append(float(line.split()[1]))

rho_rec_or = np.log10(rho_rec_or)
ef_kin_or = np.log10(ef_kin_or)
        
wind_en   = (np.asarray(wind_en_or) - np.mean(wind_en_or)) / np.std(wind_en_or)
wind_vel  = (np.asarray(wind_vel_or) - np.mean(wind_vel_or)) / np.std(wind_vel_or)
rho_rec   = (np.asarray(rho_rec_or) - np.mean(rho_rec_or)) / np.std(rho_rec_or)
sf_ts     = (np.asarray(sf_ts_or) - np.mean(sf_ts_or)) / np.std(sf_ts_or)
ef_kin    = (np.asarray(ef_kin_or) - np.mean(ef_kin_or)) / np.std(ef_kin_or)
ef_high   = (np.asarray(ef_high_or) - np.mean(ef_high_or)) / np.std(ef_high_or)
f_re      = (np.asarray(f_re_or) - np.mean(f_re_or)) / np.std(f_re_or)

In [ ]:
name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial']

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

snap = 264
zoom = {}

for i in range(len(name_list)):
    if i<30:
        base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/LH_{:d}/hydro_output/".format(i)
    else:
        base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/"
    zoom[name_list[i]] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                            tau=tau, ns=ns, sigma8=sigma8, dm_file="snapdir_{:03d}/snapshot_{:03d}".format(snap,snap), use_ids=True, numpart=4320)


In [ ]:
# Load the Halo Selection
with open("/cosmos_storage/simulations/TNG_Family/MN5_resims/resims_info/hydro_halo_sel_1pmbin.txt") as f:
    final_sel = []

    for line in f.readlines():
        final_sel.append(int(line.split()[0]))

final_sel = np.array(final_sel)

# Perform the cross-match with MTNG halos
xmatch = {}

for i in range(len(name_list)):
    xmatch[name_list[i]] = utils.cross_match(zoom[name_list[i]], snap=264, name=name_list[i])

# Load MTNG and get the fraction of halos to do the upweighting
mtng = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/TNG_Family/MTNG/", snap=264)

m200b = np.log10(1e10 * mtng.fof['halo_m200b'])

mbins = np.concatenate(
    (np.arange(11, 11.5, 0.0025),
    np.arange(11.5, 12.5, 0.005),
    np.arange(12.5, 13.5, 0.025),
    np.arange(13.5, 15.01, 0.125))
)

h_frac = np.zeros(len(final_sel))
for m in range(len(mbins)-1):
    h_frac[m] = np.where(( m200b[final_sel]>=mbins[m]) & ( m200b[final_sel]<mbins[m+1]))[0].shape[0] / \
             np.where(( m200b>=mbins[m]) & ( m200b<mbins[m+1]))[0].shape[0]

zoom_split = {}
zoom_sel = {}
for i in range(len(name_list)):
    zoom_split[name_list[i]] = utils.split_halos(zoom[name_list[i]])

    zoom_sel[name_list[i]] = {}

    zoom_sel[name_list[i]]['sel'] = xmatch[name_list[i]]['ind'][:,np.newaxis,np.newaxis]
    zoom_sel[name_list[i]]['h_frac'] = h_frac[np.newaxis, :]

In [ ]:
sSFR_mstar = {}

for i in range(30,31):
    sSFR_mstar[name_list[i]] = zoom_split[name_list[i]].sSFR_mstar(sel_mask=zoom_sel[name_list[i]], nbins=20)

In [ ]:
fig, ax = plt.subplots(dpi=200)

ax.set_xscale('log')
ax.set_yscale('log')

ax.plot(sSFR_mstar['fiducial']['mstar_mean'], sSFR_mstar['fiducial']['sSFR_mean'], label='fiducial')

In [ ]:
# Load sSFR
Nbins_sSFR = 10

zoom_sSFR = {}
for i in range(len(name_list)):
    zoom_sSFR[name_list[i]] = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/results/sSFR/sSFR_{}_Nbins{:d}.npy".format(name_list[i], Nbins_sSFR), allow_pickle=True)[0]


In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('log')

ax.set_ylabel("sSFR")
ax.set_xlabel(r"$M_{*}$ [$M_\odot$]")

for i in range(len(name_list)-1):
    ax.plot(zoom_sSFR[name_list[i]]['mstar_mean'], zoom_sSFR[name_list[i]]['sSFR_mean'], color='gray', alpha=0.5, lw=1)

ax.plot(zoom_sSFR['fiducial']['mstar_mean'], zoom_sSFR['fiducial']['sSFR_mean'], label='Fiducial', color='C3', lw=2)

ax.legend()


In [ ]:
zoom_sSFR['LH_4']['sSFR_mean']

In [ ]:
import sys
sys.path.append("/cosmos_storage/home/fgmaion/MTNG-resims/scripts")
from GP_models import SMF_Model, fgas_Model
import torch
import gpytorch

In [ ]:
model_bhmf = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_model_bhmf.pth")
likelihood_bhmf = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_likelihood_bhmf.pth")

model_bhmf.eval()
likelihood_bhmf.eval()

In [ ]:
# Define the training set
train_sel = np.arange(31)

In [ ]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    # Initialize plot
    f, ax = plt.subplots(2, 3, figsize=(15, 10), dpi=100)
    plt.subplots_adjust(wspace=0.15, hspace=0.3)

    for i in range(2):
        for j in range(3):
            test_x = torch.asarray(pars(train_sel[3*i+j], np.log10(zoom_bhmf[name_list[train_sel[3*i+j]]]['mbh'])), dtype=torch.float)
            observed_pred = likelihood_bhmf(model_bhmf(torch.asarray(test_x, dtype=torch.float)))
        
            ax[i,j].set_xlim(6,11)
            ax[i,j].set_ylim(-6.5, -1.5)

            ax[i,j].set_title('Simulation {:s}'.format(name_list[train_sel[3*i+j]]), fontsize=16)

            if j==0:
                ax[i,j].set_ylabel('$\log_{10}(\Phi/[\mathrm{Mpc}^{-3}\mathrm{dex}^{-1}])$', fontsize=16)
            
            ax[i,j].set_xlabel('$\log_{10}(M_*/M_{\odot})$', fontsize=16)

            # Get upper and lower confidence bounds
            lower, upper = observed_pred.confidence_region()

            # Shade between the lower and upper confidence bounds
            ax[i,j].fill_between(test_x[:,0].numpy(), observed_pred.mean.numpy()-observed_pred.stddev.numpy(), observed_pred.mean.numpy()+observed_pred.stddev.numpy(), color='C0', alpha=0.4, edgecolor=None)
            ax[i,j].plot(test_x[:,0], observed_pred.mean.numpy(), 'C0', lw=2, label="GP Prediction", alpha=0.9)

            # Plot test data as blue squares
            ax[i,j].plot(np.log10(zoom_bhmf[name_list[train_sel[3*i+j]]]['mbh']), np.log10(zoom_bhmf[name_list[train_sel[3*i+j]]]['bhmf']), 's', color="C0", label="Test Data")

ax[0,0].legend(loc='lower left', fontsize=12)

plt.savefig("/cosmos_storage/home/fgmaion/MTNG-resims/results/testing_plots/bhmf_test.pdf", bbox_inches='tight')

In [ ]:
import numpy as np
import torch
import gpytorch
import os
import copy

wind_en_or      = []
wind_vel_or     = []
rho_rec_or      = []
sf_ts_or        = []
ef_kin_or       = []
ef_high_or      = []
f_re_or         = []

name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial']# + ['bf_sim'] 

for i in range(len(name_list)):
    if i<30:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro_{:d}.txt".format(i)
    if i==30:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro.txt"
    if i==31:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro_bf.txt"


    with open(filename, 'r') as f:
        for line in f.readlines():
            if len(line.split())!=0:
                if line.split()[0] == 'WindEnergyIn1e51erg':
                    wind_en_or.append(float(line.split()[1]))
                if line.split()[0] == 'VariableWindVelFactor':
                    wind_vel_or.append(float(line.split()[1]))
                if line.split()[0] == 'WindFreeTravelDensFac':
                    rho_rec_or.append(float(line.split()[1]))
                if line.split()[0] == 'MaxSfrTimescale':
                    sf_ts_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackFactor':
                    ef_kin_or.append(float(line.split()[1]))
                if line.split()[0] == 'BlackHoleFeedbackFactor':
                    ef_high_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackReiorientationFactor':
                    f_re_or.append(float(line.split()[1]))
        
rho_rec_or = np.log10(rho_rec_or)
ef_kin_or  = np.log10(ef_kin_or)

wind_en   = (np.asarray(wind_en_or) - np.mean(wind_en_or)) / np.std(wind_en_or)
wind_vel  = (np.asarray(wind_vel_or) - np.mean(wind_vel_or)) / np.std(wind_vel_or)
rho_rec   = (np.asarray(rho_rec_or) - np.mean(rho_rec_or)) / np.std(rho_rec_or)
sf_ts     = (np.asarray(sf_ts_or) - np.mean(sf_ts_or)) / np.std(sf_ts_or)
ef_kin    = (np.asarray(ef_kin_or) - np.mean(ef_kin_or)) / np.std(ef_kin_or)
ef_high   = (np.asarray(ef_high_or) - np.mean(ef_high_or)) / np.std(ef_high_or)
f_re      = (np.asarray(f_re_or) - np.mean(f_re_or)) / np.std(f_re_or)

def pars(i, m2half):

    arr = np.vstack( ( m2half, np.ones(len(m2half)) * wind_en[i],\
                        np.ones(len(m2half)) * wind_vel[i],\
                        np.ones(len(m2half)) * rho_rec[i],\
                        np.ones(len(m2half)) * sf_ts[i],\
                        np.ones(len(m2half)) * ef_kin[i],\
                        np.ones(len(m2half)) * ef_high[i],\
                        np.ones(len(m2half)) * f_re[i])).T

    return arr

# Load Stellar-Mass to Halo-Mass Relation
Nbins_sSFR = 10

zoom_sSFR = {}
for i in range(len(name_list)):
    zoom_sSFR[name_list[i]] = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/results/sSFR/sSFR_{}_Nbins{:d}.npy".format(name_list[i], Nbins_sSFR), allow_pickle=True)[0]

# Filter NaNs
for i in range(len(name_list)):
    mask = ~np.isnan(zoom_sSFR[name_list[i]]['sSFR_mean']) & ~np.isnan(zoom_sSFR[name_list[i]]['mstar_mean'])
    
    zoom_sSFR[name_list[i]]['sSFR_mean'] = zoom_sSFR[name_list[i]]['sSFR_mean'][mask]
    zoom_sSFR[name_list[i]]['mstar_mean'] = zoom_sSFR[name_list[i]]['mstar_mean'][mask]

# Filter infs
for i in range(len(name_list)):
    mask = ~np.isinf(np.log10(zoom_sSFR[name_list[i]]['sSFR_mean'])) & ~np.isinf(np.log10(zoom_sSFR[name_list[i]]['mstar_mean']))
    
    zoom_sSFR[name_list[i]]['sSFR_mean'] = zoom_sSFR[name_list[i]]['sSFR_mean'][mask]
    zoom_sSFR[name_list[i]]['mstar_mean'] = zoom_sSFR[name_list[i]]['mstar_mean'][mask]

# Define the training set
train_sel = np.arange(31)

pars_global = pars(train_sel[0], np.log10(zoom_sSFR[name_list[train_sel[0]]]['mstar_mean']) )
sSFR_global = np.log10(zoom_sSFR[name_list[train_sel[0]]]['sSFR_mean'])

for i in range(len(train_sel)):
    mstar = np.log10(zoom_sSFR[name_list[train_sel[i]]]['mstar_mean'])

    arr = pars(train_sel[i], mstar)

    pars_global = np.vstack((pars_global, arr))

    sSFR_global = np.hstack((sSFR_global, np.log10(zoom_sSFR[name_list[train_sel[i]]]['sSFR_mean'])))
    
train_x = torch.asarray(pars_global, dtype=torch.float)
train_y = torch.asarray(sSFR_global, dtype=torch.float)

In [ ]:
train_y